# The empirical semivariogram: explore the distance pattern

A semivariogram compares pairs of observations. As the distance between two locations changes, do their measured values tend to become more different?

This lab creates a small synthetic dataset and uses the repository's tested Matheron estimator. Change the lag bins and maximum distance to see how the summary responds.

**You need:** Python, NumPy, Matplotlib and ipywidgets.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, interact

current = Path.cwd().resolve()
repo_root = next((p for p in (current, *current.parents)
                  if (p / 'implementations/python/geostatistics').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from a checkout of geo-dev-hub/spatial-analysis.')
sys.path.insert(0, str(repo_root / 'implementations/python/geostatistics'))
from interpolation import empirical_variogram

rng = np.random.default_rng(2026)
coordinates = rng.uniform(0, 10, size=(45, 2))
x, y = coordinates[:, 0], coordinates[:, 1]
observations = np.sin(x / 2) + np.cos(y / 3) + rng.normal(0, 0.18, size=len(x))
samples = [(float(px), float(py), float(value)) for (px, py), value in zip(coordinates, observations)]

print('Repository:', repo_root)

## 1. Inspect the observations

The values come from a smooth synthetic spatial pattern with a little random noise. This gives us a controlled example in which nearby values tend to be related.

The values are illustrative; they are not measurements from a real environmental process.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(x, y, c=observations, s=75, edgecolor="black")
fig.colorbar(sc, ax=ax, label="Synthetic observed value")
ax.set(xlabel="x coordinate", ylabel="y coordinate", title="Synthetic spatial observations")
ax.set_aspect("equal")
plt.show()

## 2. Calculate pair distances and squared differences

Each unordered pair contributes once. We group pairs by distance and calculate the average squared difference divided by two.

In [ ]:
def summarize_bins(edges):
    summaries = empirical_variogram(samples, edges)
    lags = np.asarray([row[0] for row in summaries], dtype=float)
    semivariance = np.asarray([np.nan if row[1] is None else row[1] for row in summaries], dtype=float)
    pair_counts = np.asarray([row[2] for row in summaries], dtype=int)
    return lags, semivariance, pair_counts

edges = np.linspace(0, 7, 9)
lags, semivariance, pair_counts = summarize_bins(edges)
print('Mean pair distance (empty-bin midpoint) | semivariance | pairs')
for lag, gamma, count in zip(lags, semivariance, pair_counts):
    value = '—' if np.isnan(gamma) else f'{gamma:.3f}'
    print(f'{lag:>14.2f} | {value:>12} | {count:>5d}')

## 3. Change the lag bins

The controls change the maximum distance included and the number of bins. Watch both the empirical semivariance and the number of pairs behind each estimate.

A bin with few pairs is less stable. Empty bins are left visible rather than filled with invented values.

In [ ]:
def explore(max_lag=7.0, number_of_bins=8):
    edges = np.linspace(0, max_lag, number_of_bins + 1)
    lags, gamma, counts = summarize_bins(edges)

    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    axes[0].plot(lags, gamma, marker="o", linewidth=1.5)
    axes[0].set_ylabel("Semivariance")
    axes[0].set_title("Matheron empirical semivariogram")
    axes[0].grid(alpha=0.25)

    axes[1].bar(lags, counts, width=(edges[1] - edges[0]) * 0.8)
    axes[1].set_xlabel("Mean pair distance (coordinate units; empty-bin midpoint)")
    axes[1].set_ylabel("Number of pairs")
    axes[1].grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

    print(f"Pairs included: {int(counts.sum())} of {len(coordinates) * (len(coordinates) - 1) // 2}")
    print("Empty bins are not estimates. Inspect pair counts before interpreting the curve.")

interact(
    explore,
    max_lag=FloatSlider(value=7.0, min=2.0, max=14.0, step=0.5, description="Max lag"),
    number_of_bins=IntSlider(value=8, min=3, max=16, step=1, description="Bins"),
);

## 4. Questions to investigate

1. What happens to the number of pairs as the maximum lag increases?
2. How does changing the bin count affect the shape?
3. Which bins have enough pairs to feel reasonably stable?
4. Why should an empty bin not be connected to its neighbours as if it contained a measured value?
5. What would a directional semivariogram reveal that this omnidirectional one cannot?

## 5. Important limits

This is an omnidirectional Matheron estimator using planar Euclidean distances. It does not fit a theoretical variogram or estimate a nugget, sill or range. It also does not account for trend, anisotropy or spatially structured sampling.

Continue with the [semivariogram method record](../methods/empirical-semivariogram.md), [worked exercise](../exercises/empirical-semivariogram.md) and [tested Python implementation](../implementations/python/geostatistics/README.md).